In [23]:
import pandas as pd
import os
import glob

In [31]:
# 1. EXTRAÇÃO (Extract)
print("1. A ler os ficheiros VRA de 2026...")
caminho_arquivos= 'Dados_Brutos_ANAC-2026/*.csv'
lista_arquivos = glob.glob(caminho_arquivos)

print(f"Ficheiros encontrados: {lista_arquivos}")

df_list = []
for ficheiro in lista_arquivos:
    df_temp = pd.read_csv(ficheiro, sep=';', encoding='utf-8-sig', low_memory=False)
    df_list.append(df_temp)

df_bruto = pd.concat(df_list, ignore_index=True)
print(f"Total de registos carregados: {len(df_bruto)} linhas.")

# 2. TRANSFORMAÇÃO E LIMPEZA (Transform)
print("2. A padronizar colunas e a limpar dados nulos...")
df_bruto.columns = (
    df_bruto.columns
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.replace(' ', '_')
    .str.lower()
)

colunas_uteis = [
    'sigla_icao_empresa_aerea', 'empresa_aerea', 'numero_voo',
    'sigla_icao_aeroporto_origem', 'descricao_aeroporto_origem',
    'partida_prevista', 'partida_real',
    'sigla_icao_aeroporto_destino', 'descricao_aeroporto_destino',
    'chegada_prevista', 'chegada_real', 'situacao_voo'
]
df_limpo = df_bruto[colunas_uteis].copy()

df_limpo = df_limpo.dropna(subset=['sigla_icao_aeroporto_origem', 'sigla_icao_aeroporto_destino'])

# 3. MODELAGEM RELACIONAL (Star Schema)
print("3. A construir as Tabelas Dimensão...")

dim_empresas = df_limpo[['sigla_icao_empresa_aerea', 'empresa_aerea']].drop_duplicates().reset_index(drop=True)
dim_empresas.rename(columns={'sigla_icao_empresa_aerea': 'id_empresa_icao', 'empresa_aerea': 'nome_empresa'}, inplace=True)

origens = df_limpo[['sigla_icao_aeroporto_origem', 'descricao_aeroporto_origem']].rename(
    columns={'sigla_icao_aeroporto_origem': 'id_aeroporto_icao', 'descricao_aeroporto_origem': 'nome_aeroporto'}
)
destinos = df_limpo[['sigla_icao_aeroporto_destino', 'descricao_aeroporto_destino']].rename(
    columns={'sigla_icao_aeroporto_destino': 'id_aeroporto_icao', 'descricao_aeroporto_destino': 'nome_aeroporto'}
)
dim_aeroportos = pd.concat([origens, destinos]).drop_duplicates(subset=['id_aeroporto_icao']).reset_index(drop=True)

print("4. A construir a Tabela Fato...")

fato_viagens = df_limpo[[
    'sigla_icao_empresa_aerea', 'numero_voo',
    'sigla_icao_aeroporto_origem', 'sigla_icao_aeroporto_destino',
    'partida_prevista', 'partida_real',
    'chegada_prevista', 'chegada_real', 'situacao_voo'
]].copy()

fato_viagens.rename(columns={
    'sigla_icao_empresa_aerea': 'id_empresa_icao',
    'sigla_icao_aeroporto_origem': 'id_aeroporto_origem',
    'sigla_icao_aeroporto_destino': 'id_aeroporto_destino'
}, inplace=True)

colunas_data = ['partida_prevista', 'partida_real', 'chegada_prevista', 'chegada_real']
for col in colunas_data:
    fato_viagens[col] = pd.to_datetime(fato_viagens[col], format='%d/%m/%Y %H:%M', errors='coerce')

# 4. CARGA (Load)
print("5. A exportar os ficheiros CSV saneados...")
dim_empresas.to_csv('dim_empresas.csv', index=False, encoding='utf-8')
dim_aeroportos.to_csv('dim_aeroportos.csv', index=False, encoding='utf-8')
fato_viagens.to_csv('fato_viagens.csv', index=False, encoding='utf-8')

print("ETL Concluído com sucesso! Os ficheiros limpos estão prontos para o SQL.")

1. A ler os ficheiros VRA de 2026...
Ficheiros encontrados: ['Dados_Brutos_ANAC-2026\\VRA_2026_01.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_02.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_03.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_04.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_05.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_06.csv', 'Dados_Brutos_ANAC-2026\\VRA_2026_08.csv']
Total de registos carregados: 589993 linhas.
2. A padronizar colunas e a limpar dados nulos...
3. A construir as Tabelas Dimensão...
4. A construir a Tabela Fato...
5. A exportar os ficheiros CSV saneados...
ETL Concluído com sucesso! Os ficheiros limpos estão prontos para o SQL.
